# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [2]:
import pandas as pd

reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")

reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


👀 20 adet düşük puan alan yorumdan oluşan bir örneklem seçin (rastgele) ve bunu bir listeye dönüştürün.

In [ ]:
one_star = reviews[
    (reviews["review_score"] == 1) &
    (reviews["review_comment_message"].notna())
]

sample_reviews = one_star.sample(100, random_state=42)

texts = sample_reviews["review_comment_message"].tolist()

len(texts)

20

🗣 Bu göreve başlamadan önce önceden yüklediğiniz `google_translator` paketini kullanarak bu yorumları çevirin:

In [4]:
from deep_translator import GoogleTranslator
import pandas as pd
import time

translator = GoogleTranslator(
    source="pt",
    target="en"
)

translated_reviews = []

for i, text in enumerate(texts):

    try:
        # Boş metin kontrolü
        if pd.isna(text) or str(text).strip() == "":
            translated = None

        else:
            translated = translator.translate(str(text))

        translated_reviews.append({
            "original_text": text,
            "translated_text": translated
        })

        # Google limitlerine takılmamak için bekleme
        time.sleep(1)

    except Exception as e:
        print(f"Translation error at index {i}: {e}")

        translated_reviews.append({
            "original_text": text,
            "translated_text": None
        })


translated_df = pd.DataFrame(translated_reviews)

translated_df.head(20)

,original_text,translated_text
0,Foi enviado produto Diferente daquele do anúnc...,A different product than the one in the ad was...
1,Não comprem desta loja! Tentei fazer o cancela...,Don't buy from this store! I tried to cancel t...
2,Excelente,Excellent
3,O jogo de bolas de bilhar que comprei é totalm...,The set of billiard balls that I purchased is ...
4,material ruim. sem garantia,bad material. no warranty
5,Gostaria de saber por que o meu produto consta...,I would like to know why my product is listed ...
6,Olá queria informar que o produto não foi entr...,"Hello, I wanted to inform you that the product..."
7,Já se passaram quase um mês e nada preciso da ...,It's been almost a month and I don't need your...
8,Falta receber o protetor de porta e os jogos d...,Missing the door protector and single bed sets
9,Boa tarde. \r\nNão recebo todos os produtos fa...,Good afternoon. \r\nI don't receive all the pr...


In [6]:
translated_df

,original_text,translated_text
0,Foi enviado produto Diferente daquele do anúnc...,A different product than the one in the ad was...
1,Não comprem desta loja! Tentei fazer o cancela...,Don't buy from this store! I tried to cancel t...
2,Excelente,Excellent
3,O jogo de bolas de bilhar que comprei é totalm...,The set of billiard balls that I purchased is ...
4,material ruim. sem garantia,bad material. no warranty
5,Gostaria de saber por que o meu produto consta...,I would like to know why my product is listed ...
6,Olá queria informar que o produto não foi entr...,"Hello, I wanted to inform you that the product..."
7,Já se passaram quase um mês e nada preciso da ...,It's been almost a month and I don't need your...
8,Falta receber o protetor de porta e os jogos d...,Missing the door protector and single bed sets
9,Boa tarde. \r\nNão recebo todos os produtos fa...,Good afternoon. \r\nI don't receive all the pr...


**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?
